# Prepare a model-ready inference dataset

Notebook này tạo dữ liệu cho **Protocol A** hoặc **Protocol B**. Đầu ra là sáu
GeoTIFF đã đồng đăng ký và `inference_dataset.json`. Notebook không train và
không load checkpoint.

- Protocol A: cùng Xuân Thủy, thời gian mới.
- Protocol B: vùng mới, mùa tương đương.

Phải chọn chính xác sáu STAC item ID. Không tự động lấy ảnh có cloud cover thấp
nhất rồi xem đó là quyết định khoa học.

## 1. Mount Drive và cài thư viện chuẩn bị dữ liệu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pystac-client', 'planetary-computer', 'geopandas>=1,<2',
    'rasterio==1.5.2', 'shapely', 'pyproj', 'pandas==2.2.3', 'matplotlib>=3.8,<4'
])


## 2. Khai báo Protocol A hoặc B và AOI

In [ ]:
from pathlib import Path

PROTOCOL = 'A_temporal_transfer'  # or 'B_spatial_transfer'
DATASET_ID = 'REPLACE_DATASET_ID'
REGION_ID = 'xuan_thuy'  # replace for Protocol B
AOI_PATH = Path('/content/drive/MyDrive/Prithvi_Colab/data/inference/REPLACE/aoi/aoi.geojson')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Prithvi_Colab/data/inference/REPLACE')

SEARCH_START = 'YYYY-MM-DD'
SEARCH_END = 'YYYY-MM-DD'
TARGET_CRS = 'EPSG:32648'  # Confirm the correct projected CRS for the AOI.
TARGET_RESOLUTION = 10
MAX_SCENE_CLOUD = 40

assert PROTOCOL in {'A_temporal_transfer', 'B_spatial_transfer'}
assert AOI_PATH.is_file(), AOI_PATH
print(PROTOCOL, DATASET_ID, REGION_ID, OUTPUT_ROOT)


## 3. Query candidates và duyệt metadata

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import mapping
from pystac_client import Client
import planetary_computer

aoi = gpd.read_file(AOI_PATH)
if aoi.crs is None:
    raise ValueError('AOI has no CRS')
aoi_4326 = aoi.to_crs('EPSG:4326')
search_geometry = mapping(aoi_4326.geometry.union_all())

catalog = Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(
    collections=['sentinel-2-l2a'],
    intersects=search_geometry,
    datetime=f'{SEARCH_START}/{SEARCH_END}',
    query={'eo:cloud_cover': {'lt': MAX_SCENE_CLOUD}},
)
items = list(search.items())
candidates = pd.DataFrame([
    {
        'item_id': item.id,
        'date': item.datetime.date().isoformat(),
        'cloud_cover_scene_percent': item.properties.get('eo:cloud_cover'),
        'processing_baseline': item.properties.get('s2:processing_baseline'),
        'platform': item.properties.get('platform'),
        'tile': item.properties.get('s2:mgrs_tile'),
    }
    for item in items
]).sort_values(['date', 'cloud_cover_scene_percent'])
display(candidates)
print('Candidate count:', len(candidates))


## 4. Chốt sáu item ID và quyết định offset

In [ ]:
SELECTED_ITEM_IDS = [
    'REPLACE_ITEM_01',
    'REPLACE_ITEM_02',
    'REPLACE_ITEM_03',
    'REPLACE_ITEM_04',
    'REPLACE_ITEM_05',
    'REPLACE_ITEM_06',
]
TARGET_DATE = 'YYYY-MM-DD'

# Set True only after reviewing processing_baseline and confirming that the
# exported rasters retain the raw DN convention expected by the -1000 reader.
OFFSET_VERIFIED_FOR_SOURCE_PRODUCT = False
OFFSET_VERIFICATION_NOTE = 'REPLACE with the metadata/export evidence used for this decision'

if len(SELECTED_ITEM_IDS) != 6 or len(set(SELECTED_ITEM_IDS)) != 6:
    raise ValueError('Select exactly six unique STAC item IDs')
selected = [item for item in items if item.id in SELECTED_ITEM_IDS]
if len(selected) != 6:
    missing = sorted(set(SELECTED_ITEM_IDS) - {item.id for item in selected})
    raise ValueError(f'Selected items were not returned by the query: {missing}')
selected = sorted(selected, key=lambda item: item.datetime)
selected_dates = [item.datetime.date().isoformat() for item in selected]
if selected_dates[-1] != TARGET_DATE:
    raise ValueError('TARGET_DATE must equal the final selected acquisition date')
display(candidates[candidates.item_id.isin(SELECTED_ITEM_IDS)].sort_values('date'))
print('Selected dates:', selected_dates)


## 5. Tạo common 10 m grid và xuất sáu GeoTIFF

In [ ]:
import hashlib
import json
import math
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT

BANDS = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
rasters_dir = OUTPUT_ROOT / 'rasters'
rasters_dir.mkdir(parents=True, exist_ok=True)

aoi_projected = aoi.to_crs(TARGET_CRS)
aoi_geometry = aoi_projected.geometry.union_all()
minx, miny, maxx, maxy = aoi_geometry.bounds
left = math.floor(minx / TARGET_RESOLUTION) * TARGET_RESOLUTION
bottom = math.floor(miny / TARGET_RESOLUTION) * TARGET_RESOLUTION
right = math.ceil(maxx / TARGET_RESOLUTION) * TARGET_RESOLUTION
top = math.ceil(maxy / TARGET_RESOLUTION) * TARGET_RESOLUTION
width = int(round((right - left) / TARGET_RESOLUTION))
height = int(round((top - bottom) / TARGET_RESOLUTION))
transform = from_origin(left, top, TARGET_RESOLUTION, TARGET_RESOLUTION)
inside_aoi = geometry_mask(
    [aoi_geometry.__geo_interface__],
    out_shape=(height, width),
    transform=transform,
    invert=True,
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

image_records = []
for item in selected:
    acquisition_date = item.datetime.date().isoformat()
    output_path = rasters_dir / f'Sentinel2_{acquisition_date}_{REGION_ID}_6bands_10m.tif'
    stack = np.zeros((6, height, width), dtype=np.uint16)
    valid = inside_aoi.copy()
    for band_index, band_name in enumerate(BANDS):
        if band_name not in item.assets:
            raise KeyError(f'{band_name} is missing from STAC item {item.id}')
        with rasterio.open(item.assets[band_name].href) as source:
            with WarpedVRT(
                source,
                crs=TARGET_CRS,
                transform=transform,
                width=width,
                height=height,
                resampling=Resampling.bilinear,
                nodata=0,
            ) as vrt:
                band = vrt.read(1, masked=True, out_dtype='float32')
                band_valid = ~np.ma.getmaskarray(band) & np.isfinite(band.filled(np.nan))
                valid &= band_valid
                values = np.rint(band.filled(0)).clip(0, np.iinfo(np.uint16).max).astype(np.uint16)
                stack[band_index] = values
    stack[:, ~valid] = 0
    profile = {
        'driver': 'GTiff', 'height': height, 'width': width, 'count': 6,
        'dtype': 'uint16', 'crs': TARGET_CRS, 'transform': transform,
        'nodata': 0, 'compress': 'deflate', 'tiled': True,
    }
    with rasterio.open(output_path, 'w', **profile) as destination:
        destination.write(stack)
        destination.write_mask(valid.astype(np.uint8) * 255)
        for index, description in enumerate(BANDS, start=1):
            destination.set_band_description(index, description)
        destination.update_tags(
            acquisition_date=acquisition_date,
            stac_item_id=item.id,
            processing_baseline=str(item.properties.get('s2:processing_baseline', 'unknown')),
            export_note='Raw source DN reprojected to common 10 m grid; reader offset is not applied here',
        )
    image_records.append({
        'path': str(output_path.relative_to(OUTPUT_ROOT)).replace('\\', '/'),
        'date': acquisition_date,
        'sha256': sha256_file(output_path),
        'stac_item_id': item.id,
    })
    print(output_path.name, int(valid.sum()))

print('Grid:', TARGET_CRS, height, width, transform)


## 6. Visual QA cho target-date raster

In [ ]:
import matplotlib.pyplot as plt

target_path = OUTPUT_ROOT / image_records[-1]['path']
with rasterio.open(target_path) as source:
    rgb = source.read([3, 2, 1]).astype(np.float32)
    mask = source.dataset_mask() > 0
for channel in range(3):
    values = rgb[channel][mask]
    low, high = np.percentile(values, [2, 98])
    rgb[channel] = np.clip((rgb[channel] - low) / max(high - low, 1), 0, 1)
rgb[:, ~mask] = 0
plt.figure(figsize=(10, 8))
plt.imshow(np.moveaxis(rgb, 0, -1))
plt.title(f'{REGION_ID} target image {TARGET_DATE}')
plt.axis('off')
plt.show()


## 7. Ghi inference manifest sau khi offset được xác nhận

In [ ]:
if not OFFSET_VERIFIED_FOR_SOURCE_PRODUCT:
    raise RuntimeError(
        'Stop: review source processing_baseline and export radiometry, then set '
        'OFFSET_VERIFIED_FOR_SOURCE_PRODUCT=True with a verification note.'
    )

manifest = {
    'schema': 'prithvi-inference-dataset-v1',
    'protocol': PROTOCOL,
    'dataset_id': DATASET_ID,
    'region_id': REGION_ID,
    'target_date': TARGET_DATE,
    'images': [{key: record[key] for key in ['path', 'date', 'sha256']} for record in image_records],
    'bands': BANDS,
    'grid': {
        'crs': TARGET_CRS,
        'shape': [height, width],
        'resolution_metres': TARGET_RESOLUTION,
        'same_region_as_training': PROTOCOL == 'A_temporal_transfer',
    },
    'radiometry': {
        'dn_offset': -1000.0,
        'scale': 10000.0,
        'offset_verified_for_source_product': True,
        'verification_note': OFFSET_VERIFICATION_NOTE,
    },
    'normalization': {
        'mean': [1087, 1342, 1433, 2734, 1958, 1363],
        'std': [2248, 2179, 2178, 1850, 1242, 1049],
    },
    'inference': {
        'patch_size': 224,
        'stride': 112,
        'overlap_blending': 'mean_softmax_probability',
    },
    'evaluation': {
        'date_matched_ground_truth': None,
        'status': 'qualitative until an independent date-matched label set is supplied',
    },
    'provenance': {
        'aoi_source': str(AOI_PATH),
        'stac_collection': 'sentinel-2-l2a',
        'selected_item_ids': SELECTED_ITEM_IDS,
    },
}
manifest_path = OUTPUT_ROOT / 'inference_dataset.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Manifest:', manifest_path)
print('Manifest SHA-256:', sha256_file(manifest_path))
